# Analyse du dédoublonnage

## Fonction d'analyse
Calcul et restitution du nombre de ligne en défut d'intégrité

In [1]:
from datetime import datetime
import json
from tab_dataset import Cdataset
import pandas as pd
import ntv_pandas as npd
import pathlib

def analyse_integrite(data, schema, affiche=True, indic=True):
    '''analyse les relations du DataFrame 'data' définies dans le schéma 'schema'.
    Le nombre de lignes en erreur par relation (dict) est retourné et optionnellement affiché (paramètre 'affiche=True') . 
    Les lignes en erreur sont optionnellement ajoutées (paramètre 'indic=True') à 'data' sous forme de champs booléens par relation.
    '''
    dic_errors = Cdataset(data).check_relationship(schema)
    dic_count = {name: len(errors) for name, errors in dic_errors.items()}
    if affiche:
        for name, total in dic_count.items():
            print('{:<50} {:>5}'.format(name, total))
    if indic:
        data['ok'] = True
        for name, errors in dic_errors.items():
            data[name] = True
            data.loc[errors, name] = False
            data['ok'] = data['ok'] & data[name] 
        if affiche:
            nb_ok = sum(data['ok'])
            nb_ko = len(data) - sum(data['ok'])      
            print("\nnombre d'enregistrements sans erreurs : ", nb_ok)
            print("nombre d'enregistrements avec au moins une erreur : ", nb_ko)
            print("dont doublons : ", dic_count['index - id_pdc_itinerance'])
            print("\ntaux d'erreur : ", round(nb_ko / len(data) * 100), ' %')
    return dic_count

## Schéma de données
Le schéma de données restreint à la propriété 'relationship' et construit à partir du modèle de données est le suivants :

In [2]:
# complément à inclure dans le schéma de données
schema = {
    'relationships': [
         # relation unicité des pdl
         {"fields": ["id_pdc_itinerance", "index"],                    "link" : "coupled" },   
         # relations inter entités
         {"fields": ["id_station_itinerance", "contact_operateur"],    "link" : "derived" },
         {"fields": ["id_station_itinerance", "nom_enseigne"],         "link" : "derived" },
         {"fields": ["id_station_itinerance", "coordonneesXY"],        "link" : "derived" },
         {"fields": ["id_pdc_itinerance", "id_station_itinerance"],    "link" : "derived" },
         # relations intra entité - station
         {"fields": ["id_station_itinerance", "nom_station"],          "link" : "derived" },
         {"fields": ["id_station_itinerance", "implantation_station"], "link" : "derived" },
         #{"fields": ["id_station_itinerance", "date_maj"],             "link" : "derived" },
         {"fields": ["id_station_itinerance", "nbre_pdc"],             "link" : "derived" },
         {"fields": ["id_station_itinerance", "condition_acces"],      "link" : "derived" },
         {"fields": ["id_station_itinerance", "horaires"],             "link" : "derived" },
         {"fields": ["id_station_itinerance", "station_deux_roues"],   "link" : "derived" },
         # relations intra entité - localisation
         {"fields": ["coordonneesXY", "adresse_station"],              "link" : "derived" }
    ]
}

## Initialisation des données
Fichier pandas

In [3]:
origine = 'datagouv_organization_or_owner'
priorite = 'priorite'
coord = 'coordonneesXY'
id_station = 'id_station_itinerance'
id_pdc = 'id_pdc_itinerance'
last_modif = 'last_modified'
date_maj = 'date_maj'

unicite_stations = [id_station, origine, date_maj, last_modif]
filtre = [priorite,  date_maj, last_modif]
id_station_pdc = [id_station, id_pdc]

In [4]:
file_irve_brut = 'consolidation-etalab-schema-irve-statique-v-2.3.1-20260301.csv'
file_irve = 'consolidation-etalab-schema-irve-statique-v-2.3.1-20260401.csv'

irve_brut = pd.read_csv(file_irve_brut, sep=',', low_memory=False, dtype='object').reset_index()
irve_brut[last_modif] = irve_brut['datagouv_last_modified']

irve = pd.read_csv(file_irve, sep=',', low_memory=False, dtype='object').reset_index()
irve[last_modif] = irve['datagouv_last_modified']

## Données brutes

In [21]:
print('nombre de lignes : {}, nombre de pdc : {} \n'.format(len(irve_brut), len(irve_brut.groupby([id_pdc]).count())))
print(irve_brut.groupby([origine]).count()['index'].sort_values(ascending=False)[0:10], '\n')
res_brut = analyse_integrite(irve_brut, schema)

nombre de lignes : 379946, nombre de pdc : 147826 

datagouv_organization_or_owner
QualiCharge                       61545
Engie Mobilités Electriques       58305
GIREVE                            31714
TotalEnergies Marketing France    29490
Mobilize Power Solutions          21033
IZIVIA                            15767
Driveco                           15551
ubitricity                        15349
Alizé                             14334
STATIONS-E                        13260
Name: index, dtype: int64 

index - id_pdc_itinerance                          295800
contact_operateur - id_station_itinerance          78950
nom_enseigne - id_station_itinerance               61760
coordonneesXY - id_station_itinerance              117807
id_station_itinerance - id_pdc_itinerance          150304
nom_station - id_station_itinerance                57457
implantation_station - id_station_itinerance       79455
nbre_pdc - id_station_itinerance                   73233
condition_acces - id_station_i

## Dédoublonnage actuel

In [6]:
print('nombre de lignes : {} \n'.format(len(irve)))
print(irve.groupby([origine]).count()['index'].sort_values(ascending=False)[0:10], '\n')
res_actuel = analyse_integrite(irve, schema)

nombre de lignes : 148286 

datagouv_organization_or_owner
QualiCharge                                  46503
GIREVE                                       31714
Alizé                                        13624
IZIVIA                                       12958
Eco-Movement                                 10806
Indigo Group                                  6907
Driveco                                       3935
TotalEnergies Marketing France                1849
Engie Mobilités Electriques                   1827
Citeos Ingénierie IdF & Est (Cogelum IdF)     1485
Name: index, dtype: int64 

index - id_pdc_itinerance                            816
contact_operateur - id_station_itinerance           1777
nom_enseigne - id_station_itinerance                4064
coordonneesXY - id_station_itinerance               5402
id_station_itinerance - id_pdc_itinerance             28
nom_station - id_station_itinerance                 2436
implantation_station - id_station_itinerance        2408
nbre

## Dédoublonnage proposé

In [7]:
stations = irve_brut.drop_duplicates(unicite_stations).copy()

len(stations), len(irve_brut.drop_duplicates(id_station))

(126827, 59045)

### Dédoublonnage direct station

In [8]:
stations[priorite] = stations[origine] == 'QualiCharge'
stat_direct = stations.sort_values(by=[id_station] + filtre).drop_duplicates(id_station, keep='last').copy()

In [9]:
print(stations[priorite].sum(), stat_direct[priorite].sum(), len(stat_direct))
stat_direct.groupby([origine]).count()['index'].sort_values(ascending=False)[0:15]

13597 13597 59045


datagouv_organization_or_owner
QualiCharge                                  13597
Eco-Movement                                 10485
GIREVE                                        9873
IZIVIA                                        6656
Alizé                                         4483
GREENEA                                       1910
Load Stations                                 1091
Driveco                                        892
Syndicat Départemental d'Energie du Tarn       700
Citeos Ingénierie IdF & Est (Cogelum IdF)      616
ZE-WATT                                        607
SOREGIES                                       451
Electric 55 Charging                           391
ZEborne                                        377
Mobilize Power Solutions                       373
Name: index, dtype: int64

### Dédoublonnage direct pdc

In [10]:
pdc_stat = stat_direct[unicite_stations + [priorite]].merge(irve_brut, how='left', on=unicite_stations)
pdc_stat_unique = pdc_stat.sort_values(by=id_station_pdc + filtre).drop_duplicates(id_station_pdc, keep='last').copy()
pdc_direct =  pdc_stat_unique.sort_values(by=[id_pdc] + filtre).drop_duplicates(id_pdc, keep='last').copy()
del(pdc_direct['index'])
pdc_direct = pdc_direct.reset_index()

### Bilan dédoublonnage direct

In [11]:
print(len(pdc_stat), len(pdc_stat_unique), len(pdc_direct), '\n')
print(pdc_direct.groupby([origine]).count()['index'].sort_values(ascending=False)[0:10], '\n')
res_direct = analyse_integrite(pdc_direct, schema)

172995 172424 144823 

datagouv_organization_or_owner
QualiCharge                       61545
GIREVE                            22430
Alizé                             13576
IZIVIA                            12241
Indigo Group                       6907
Eco-Movement                       3912
Driveco                            3460
TotalEnergies Marketing France     1848
Engie Mobilités Electriques        1745
Electric 55 Charging               1291
Name: index, dtype: int64 

index - id_pdc_itinerance                              0
contact_operateur - id_station_itinerance              2
nom_enseigne - id_station_itinerance                   7
coordonneesXY - id_station_itinerance                964
id_station_itinerance - id_pdc_itinerance              0
nom_station - id_station_itinerance                 1246
implantation_station - id_station_itinerance          54
nbre_pdc - id_station_itinerance                    1144
condition_acces - id_station_itinerance               23
horai

### Dédoublonnage indirect station

In [12]:
stations = pdc_direct.drop_duplicates([id_station]).copy()

#### Stations avec coordonnées identiques et origine différente

In [13]:
stations['dupl_coord_owner'] = ~stations.duplicated(keep=False, subset=[coord, origine])

stations_ext = stations[stations['dupl_coord_owner']].copy()
stations_int = stations[~stations['dupl_coord_owner']].copy()

stat_xy = stations_ext.sort_values(by=[coord, priorite]).drop_duplicates([coord], keep='last').copy()
stations_xy = pd.concat([stations_int, stat_xy])

In [14]:
print(stations_xy[priorite].sum(), stat_direct[priorite].sum(), len(stations_xy))
stations_xy.groupby([origine]).count()['index'].sort_values(ascending=False)[0:15]

13597 13597 44146


datagouv_organization_or_owner
QualiCharge                                  13597
IZIVIA                                        6611
GIREVE                                        6262
Alizé                                         4167
Eco-Movement                                  3826
Driveco                                        889
Syndicat Départemental d'Energie du Tarn       656
Citeos Ingénierie IdF & Est (Cogelum IdF)      615
ZE-WATT                                        607
SOREGIES                                       406
Electric 55 Charging                           391
ZEborne                                        377
Mobilize Power Solutions                       347
Rossini Energy                                 311
Engie Mobilités Electriques                    309
Name: index, dtype: int64

### Dédoublonnage indirect pdc

In [16]:
pdc_stat = stations_xy[[id_station]].merge(pdc_direct, how='left', on=id_station)

In [17]:
print(len(pdc_stat), len(pdc_direct))
print(pdc_stat[priorite].sum(), pdc_direct[priorite].sum(), '\n')
print(pdc_stat.groupby([origine]).count()['index'].sort_values(ascending=False)[0:15], '\n')
res_indirect = analyse_integrite(pdc_stat, schema)

142295 144823
61545 61545 

datagouv_organization_or_owner
QualiCharge                                  61545
GIREVE                                       20241
Alizé                                        13568
IZIVIA                                       12236
Indigo Group                                  6907
Eco-Movement                                  3911
Driveco                                       3460
TotalEnergies Marketing France                1848
Engie Mobilités Electriques                   1743
Electric 55 Charging                          1291
Qovoltis                                      1198
Citeos Ingénierie IdF & Est (Cogelum IdF)     1118
Lidl                                          1085
e-Totem                                        896
Rossini Energy                                 855
Name: index, dtype: int64 

index - id_pdc_itinerance                              0
contact_operateur - id_station_itinerance              2
nom_enseigne - id_station_itineran

## Synthèse des dédoublonnages

In [22]:
print('données brutes       : total pdc {:<7}'.format(len(irve_brut.groupby([id_pdc]).count())))
print('solution actuelle    : total pdc {:<7} avec pdc ok {:<7}, pdc avec erreur {:>5} dont doublons {}'.format(len(irve), sum(irve['ok']), len(irve) - sum(irve['ok']), res_actuel['index - id_pdc_itinerance']))
print('proposition direct   : total pdc {:<7} avec pdc ok {:<7}, pdc avec erreur {:>5} dont doublons {}'.format(len(pdc_direct), sum(pdc_direct['ok']), len(pdc_direct) - sum(pdc_direct['ok']), res_direct['index - id_pdc_itinerance']))
print('proposition indirect : total pdc {:<7} avec pdc ok {:<7}, pdc avec erreur {:>5} dont doublons {}'.format(len(pdc_stat), sum(pdc_stat['ok']), len(pdc_stat) - sum(pdc_stat['ok']), res_indirect['index - id_pdc_itinerance']))


données brutes       : total pdc 147826 
solution actuelle    : total pdc 148286  avec pdc ok 132857 , pdc avec erreur 15429 dont doublons 816
proposition direct   : total pdc 144823  avec pdc ok 135295 , pdc avec erreur  9528 dont doublons 0
proposition indirect : total pdc 142295  avec pdc ok 138221 , pdc avec erreur  4074 dont doublons 0
